In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/06_dim_encounter_type
# Purpose  : SCD Type 1 for encounter types (48 distinct codes).
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_encounter_type_changes")
def v_encounter_type_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.encounters")
    return df.select(
        F.xxhash64("encounter_code").alias("encounter_type_key"),
        "encounter_code",
        "encounter_description",
        "encounter_class",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_encounter_type")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_encounter_type",
    source="v_encounter_type_changes",
    keys=["encounter_code"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)